# RAG Minecraft

## Configuration Gemini API

In [5]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import csv
import os
import pickle
import re
import shutil
import time
import uuid
import json
from pathlib import Path
from urllib.parse import urlparse, unquote

import requests
import cloudscraper
from bs4 import BeautifulSoup
from IPython.display import Markdown

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate, format_document
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.storage import LocalFileStore
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_classic.retrievers import MultiVectorRetriever

from langchain_ollama import ChatOllama

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)


C:\Users\chahi\AppData\Local\Temp\ipykernel_21020\2320296692.py:25: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
from api_config import configure_google_api_key

GOOGLE_API_KEY = configure_google_api_key()

### Config sources

In [7]:
WIKI_PAGES =  "Minecraft"

FANDOM_URLS = [
    "https://minecraft.fandom.com/fr/wiki/Survie",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif",
    "https://minecraft.fandom.com/fr/wiki/Hardcore",
    "https://minecraft.fandom.com/fr/wiki/Fabrication",
    "https://minecraft.fandom.com/fr/wiki/Cuisson",
    "https://minecraft.fandom.com/fr/wiki/Alchimie",
    "https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures"
]

In [8]:
scraper = cloudscraper.create_scraper()

In [9]:
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

summarize_chain = (
    PromptTemplate.from_template(
        """
Résume cette page Minecraft pour un moteur de recherche sémantique.

Mentionne explicitement :
- les principaux concepts ;
- les créatures ;
- les objets ;
- les structures ;
- les biomes ;
- les mécaniques de jeu.

Le résumé doit faire entre 100 et 200 mots.

{doc}
"""
    )
    | llm
    | StrOutputParser()
)

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)

## Utils

#### General

In [10]:
def page_loaded(page_name: str) -> bool:
    file_name = Path(f"files/{page_name}.txt")
    return file_name.exists()

In [11]:
def write_txt(file_name, paragraphs):
    # Crée le dossier parent automatiquement s'il n'existe pas
    folder = os.path.dirname(file_name)
    if folder:
        os.makedirs(folder, exist_ok=True)

    # Sauvegarde au format JSON Lines (JSONL) pour éviter les corruptions de texte
    with open(file_name, "w", encoding="utf-8") as f:
        for p in paragraphs:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

#### Wikipedia

In [12]:
def parse_wikipedia_sections(soup, source):

    docs = []

    h2 = None
    h3 = None
    h4 = None

    current_text = []

    def save_section():
        nonlocal current_text

        text = "\n".join(current_text).strip()

        # ignorer sections vides
        if not text:
            current_text = []
            return

        title_parts = [x for x in [h2, h3, h4] if x]
        section_title = " > ".join(title_parts)
        if section_title in ["", " "]:
            section_title_title = "Introduction"

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": source,
                    "section": section_title
                }
            )
        )

        current_text = []

    for tag in soup.find_all(["h2", "h3", "h4", "p", "li"]):

        txt = tag.get_text(" ", strip=True)

        if not txt:
            continue

        # arrêt avant les sections inutiles
        if tag.name == "h2" and any(
            x in txt
            for x in [
                "Références",
                "Notes et références",
                "Voir aussi",
                "Liens externes",
                "Accueil",
                "Autres versions"
            ]
        ):
            break

        if tag.name == "h3" and any(
            x in txt
            for x in [
                "Autres versions"
            ]
        ):
            break

        if tag.name == "h2":

            save_section()

            h2 = txt
            h3 = None
            h4 = None

        elif tag.name == "h3":

            save_section()

            h3 = txt
            h4 = None

        elif tag.name == "h4":

            save_section()

            h4 = txt

        else:
            current_text.append(txt)

    save_section()

    return docs

#### Fandom

In [13]:
def clean_wiki_text(text: str) -> str:

    # Guillemets typographiques → guillemets simples
    text = text.replace("\u00ab", '"').replace("\u00bb", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2026", "...")

    # Liens wiki [[Page|texte]] → texte, [[Page]] → Page
    text = re.sub(r"\[\[(?:[^|\]]*\|)?([^\]]+)\]\]", r"\1", text)

    # Templates simples {{nom|val}} → val, {{nom}} → supprimé
    text = re.sub(r"\{\{[^}]*\|([^}]*)\}\}", r"\1", text)
    text = re.sub(r"\{\{[^}]*\}\}", "", text)

    # Balises HTML résiduelles
    text = re.sub(r"<[^>]+>", "", text)

    # Références [1], [note 2], etc.
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\[note \d+\]", "", text)

    # Marqueurs de version [Version Java 1.x]
    text = re.sub(r"\[Version [^\]]+\]", "", text)

    # Lignes qui ne contiennent que de la ponctuation ou des caractères spéciaux
    text = re.sub(r"^[^\w\s]{1,5}$", "", text, flags=re.MULTILINE)

    # Espaces multiples sur une même ligne
    text = re.sub(r"[ \t]+", " ", text)

    # Lignes vides multiples → une seule
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

def remove_empty_sections(text: str) -> str:
    """
    Supprime les titres de section qui ne sont pas suivis de contenu.
    Ex : 'SECTION: Galerie\n\nSECTION: Combat\n' → garde seulement Combat s'il a du contenu.
    """
    lines = text.split("\n")
    result = []
    i = 0

    while i < len(lines):
        line = lines[i]

        if line.startswith("SECTION:") or line.startswith("SOUS-SECTION:"):
            # Chercher si la section a du contenu
            j = i + 1
            # Sauter les lignes vides juste après le titre
            while j < len(lines) and lines[j].strip() == "":
                j += 1

            # S'il y a du contenu non-vide et que ce n'est pas une autre section
            if j < len(lines) and not (
                lines[j].startswith("SECTION:") or lines[j].startswith("SOUS-SECTION:")
            ):
                result.append(line)
            # Sinon on saute le titre de section vide
        else:
            result.append(line)

        i += 1

    return "\n".join(result)

def remove_noise_lines(text: str) -> str:
    """
    Supprime les lignes qui ne contiennent pas assez d'information :
    - moins de 20 caractères sauf si c'est un titre de section
    - lignes qui ressemblent à des noms de fichier (image, .png, .jpg)
    - lignes qui ne sont que des chiffres ou caractères spéciaux
    """
    lines = text.split("\n")
    result = []

    for line in lines:
        stripped = line.strip()

        # Toujours garder les titres de section
        if stripped.startswith("SECTION:") or stripped.startswith("SOUS-SECTION:"):
            result.append(line)
            continue

        # Supprimer les noms de fichiers image
        if re.match(r"(?i).*\.(png|jpg|jpeg|gif|svg|webp|ogg|mp3|wav)$", stripped):
            continue

        # Supprimer les lignes trop courtes (bruit)
        if len(stripped) < 20 and stripped != "":
            continue

        # Supprimer les lignes qui ne sont que des chiffres/ponctuation
        if re.match(r"^[\d\s.,;:!?()\[\]«»\"'/-]+$", stripped):
            continue

        if stripped.startswith("v d m"):
            continue

        if stripped.startswith("↑"):
            continue

        if "google.fr/search" in stripped:
            continue

        if "Version Bedrock" in stripped and len(stripped) > 100:
            continue

        if "Historique des versions" in stripped and len(stripped) > 100:
            continue

        result.append(line)

    return "\n".join(result)

STOP_HEADINGS = {
    "références", "notes et références", "notes diverses",
    "voir aussi", "liens externes", "annexes",
    "galerie", "historique", "succès",
    "créatures des autres jeux", "créatures de minecraft earth",
    "créatures de minecraft dungeons", "créatures prévues",
    "créatures inutilisées", "créatures supprimées",
    "créatures non implémentées", "créatures de la version éducation",
    "tutoriels",
}

def extract_list(tag) -> list:
    items = []
    for li in tag.find_all("li", recursive=False):
        text_parts = []
        for node in li.children:
            if not hasattr(node, "name"):
                text_parts.append(str(node))
            elif node.name not in ["ul", "ol"]:
                text_parts.append(node.get_text(" ", strip=True))
        text = clean_wiki_text(" ".join(text_parts))
        if text and len(text) >= 15:
            items.append(f"- {text}")
        for sub in li.find_all(["ul", "ol"], recursive=False):
            for sub_item in extract_list(sub):
                items.append(f"  {sub_item}")
    return items

def get_cell_name(td):
    td = BeautifulSoup(str(td), "html.parser")

    # Remplacer les liens par leur title
    for a in td.find_all("a"):
        title = a.get("title", "").strip()
        if title and "modifier" not in title.lower():
            a.replace_with(title)

    # Remplacer les images par leur alt
    for img in td.find_all("img"):
        alt = img.get("alt", "").strip()
        if alt:
            img.replace_with(alt)

    return td.get_text(" ", strip=True)

def extract_table(tag) -> list:
    rows = []

    # Récupérer les en-têtes
    headers = []
    header_row = tag.find("tr")
    if header_row:
        headers = [
            clean_wiki_text(th.get_text(" ", strip=True))
            for th in header_row.find_all("th")
            if len(th.get_text(strip=True)) >= 2
        ]

    for tr in tag.find_all("tr"):
        cells = []
        for td in tr.find_all("td"):
            cell_text = clean_wiki_text(get_cell_name(td))
            if cell_text and len(cell_text) >= 2:
                cells.append(cell_text)

        if not cells:
            continue

        if headers and len(headers) == len(cells):
            parts = [f"{h} : {c}" for h, c in zip(headers, cells)]
            rows.append("- " + " | ".join(parts))
        else:
            rows.append("- " + " : ".join(cells))

    return rows


### Scraping functions

#### Wikipedia

In [14]:
def scrape_wikipedia(title: str):

    url = "https://fr.wikipedia.org/w/api.php"

    params = {
        "action": "parse",
        "page": title,
        "format": "json",
        "prop": "text",
        "redirects": True
    }

    r = requests.get(
        url,
        params=params,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    data = r.json()

    html = data["parse"]["text"]["*"]
    soup = BeautifulSoup(html, "html.parser")
    docs = parse_wikipedia_sections(soup, f"wikipedia:{title}" )
    
    docs = [
        {
            "page_content": doc.page_content,
            "source": doc.metadata.get("source"),
            "section": doc.metadata.get("section")
        } 
        for doc in docs
    ]

    write_txt(
        f"files/wikipedia.txt",
        docs
    )
    return docs


#### Fandom

In [15]:
def scrape_fandom(url: str, page_name = str, mode:str ="windows"):

    if mode == "mac":

        r = requests.get(
           url,
           headers={"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"},
           timeout=30,
       )
        

    else:

        headers_req = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0 Safari/537.36"
            ),
            "Accept-Language": "fr-FR,fr;q=0.9",
            "Referer": "https://www.google.com/"
        }

        r = scraper.get(url, headers=headers_req)

    soup = BeautifulSoup(r.text, "html.parser")

    content = soup.find("div", {"class": "mw-parser-output"})
    if not content:
        return []

    # ==========================
    # CLEAN STATIC BLOCKS
    # ==========================
    selectors_to_remove = [
        "#toc",
        ".mw-references-wrap",
        "ol.references",
        ".navbox",
        ".vertical-navbox",
        ".portable-infobox",
        ".catlinks",
        ".mw-editsection",
        ".reference"
    ]

    for selector in selectors_to_remove:
        for element in content.select(selector):
            element.decompose()

    docs = []

    h2 = None
    h3 = None
    h4 = None

    current_content = []

    def save_section():
        nonlocal current_content

        text = "\n".join(current_content).strip()

        text = remove_empty_sections(text)
        text = remove_noise_lines(text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()

        if not text:
            current_content = []
            return

        section_parts = [x for x in [h2, h3, h4] if x]

        section_title = (
            "Introduction"
            if not section_parts
            else " > ".join(section_parts)
        )

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": url,
                    "section": section_title
                }
            )
        )

        current_content = []

    # ==========================
    # MAIN LOOP
    # ==========================
    for tag in content.children:

        if not getattr(tag, "name", None):
            continue

        # ==========================
        # HEADINGS — h1 inclus pour FAQ
        # ==========================
        heading_tag = None

        if tag.name in ["h1", "h2", "h3", "h4"]:
            heading_tag = tag
        elif tag.name == "div" and "mw-heading" in tag.get("class", []):
            heading_tag = tag.find(["h1", "h2", "h3", "h4"], recursive=False)

        if heading_tag:
            raw_heading = heading_tag.get_text(" ", strip=True)
            heading = clean_wiki_text(raw_heading)
            heading = re.sub(r"\[.*?\]", "", heading).strip()

            if not heading:
                continue

            # ==========================
            # STOP SECTIONS
            # ==========================
            if heading.lower() in STOP_HEADINGS:
                save_section()
                break

            # ==========================
            # FAQ DETECTION
            # ==========================
            b_texts = [b.get_text(strip=True).lower() for b in heading_tag.find_all("b")]
            is_faq = any(t == "q:" for t in b_texts)

            if is_faq:
                # Sauvegarder la section précédente (Q/R précédente)
                save_section()

                # Extraire le texte de la question sans le préfixe "Q:"
                question_text = clean_wiki_text(heading)
                question_text = re.sub(r"^Q\s*:\s*", "", question_text, flags=re.IGNORECASE).strip()

                # Chaque question devient sa propre section
                h2 = f"FAQ > {question_text}"
                h3 = None
                h4 = None

                # Ajouter la question en début de contenu
                current_content.append(f"Q: {question_text}")
                continue

            # ==========================
            # SECTION NORMALE
            # ==========================
            save_section()

            if heading_tag.name in ["h1", "h2"]:
                h2 = heading
                h3 = None
                h4 = None
            elif heading_tag.name == "h3":
                h3 = heading
                h4 = None
            elif heading_tag.name == "h4":
                h4 = heading

            continue

        # ==========================
        # PARAGRAPHS
        # ==========================
        if tag.name == "p":
            text = clean_wiki_text(tag.get_text(" ", strip=True))
            if not text or len(text) < 10:
                continue
            current_content.append(text)

        # ==========================
        # DL/DD — réponses FAQ et définitions
        # ==========================
        elif tag.name == "dl":
            for dd in tag.find_all("dd", recursive=True):
                raw = dd.get_text(" ", strip=True)
                # Supprimer le préfixe "R:" s'il existe
                raw = re.sub(r"^R\s*:\s*", "", raw, flags=re.IGNORECASE).strip()
                text = clean_wiki_text(raw)
                if not text or len(text) < 10:
                    continue
                current_content.append(text)

        # ==========================
        # LISTS
        # ==========================
        elif tag.name in ["ul", "ol"]:
            current_content.extend(extract_list(tag))

        # ==========================
        # TABLES
        # ==========================
        elif tag.name == "table":
            current_content.extend(extract_table(tag))

    save_section()

    page_name = unquote(url.split("/wiki/")[-1])

    print(f"FANDOM OK: {page_name} ({len(docs)} sections)")

    docs = [
        {
            "page_content": doc.page_content,
            "source": doc.metadata.get("source"),
            "section": doc.metadata.get("section")
        }
        for doc in docs
    ]

    write_txt(f"files/{page_name}.txt", docs)

    return docs

### Build dataset functions

In [16]:
dossiers_a_nettoyer = [
    "files",        # Tes pages brutes en JSONL
    "chroma_minecraft_multivec",    # La base de données Chroma
    "local_store"   # Le dossier de ton LocalFileStore (parfois nommé 'docstore' ou autre)
]

print("=== NETTOYAGE EN COURS ===")
for dossier in dossiers_a_nettoyer:
    if os.path.exists(dossier):
        try:
            shutil.rmtree(dossier)
            print(f"Dossier '{dossier}' supprimé avec succès.")
        except Exception as e:
            print(f"Erreur lors de la suppression de '{dossier}' : {e}")
            print("Pense bien à faire 'Kernel -> Restart' avant de lancer cette cellule.")
    else:
        print(f"Le dossier '{dossier}' est déjà absent (propre).")

=== NETTOYAGE EN COURS ===
Dossier 'files' supprimé avec succès.
Dossier 'chroma_minecraft_multivec' supprimé avec succès.
Le dossier 'local_store' est déjà absent (propre).


In [17]:
def scrape_web(platform : str = "windows"):

    new_pages = []

    if not page_loaded("wikipedia"):
        try:
            scrape_wikipedia(WIKI_PAGES)
            print("Chargement de wikipedia OK")
            new_pages.append("wikipedia")
        except Exception as e:
            print("Soucis avec le chargement de la page wikipedia:", e)
    else:
        print("Wikipedia deja chargee")
    
    for url in FANDOM_URLS:
        
        parsed = urlparse(url)

        page_name = unquote(
            parsed.path.split("/wiki/")[-1]
        )     

        if not page_loaded(page_name):
            
            try:

                doc = scrape_fandom(url, page_name, mode=platform)

                if doc:
                    new_pages.append(page_name)
                    print(f"Chargement de la page {page_name} OK")

            except Exception as e:
                print("Soucis avec le chargement de la page wiki:", url, e)
        
        else:
            print(f"Wiki : {page_name} deja chargee")

    return new_pages

In [18]:
def load_txt_documents(page_names: list[str]):
    documents = []
    
    for page in page_names:
        file_path = f"files/{page}.txt"
        print(f"Lecture du fichier : {file_path}")
        
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    
                    # Lecture directe du JSON de la ligne
                    data = json.loads(line)
                    
                    metadata = {
                        "source": data.get("source", ""),
                        "section": data.get("section", ""),
                        "file_origin": file_path
                    }
                    
                    doc = Document(
                        page_content=data.get("page_content", ""), 
                        metadata=metadata
                    )
                    documents.append(doc)
                            
        except Exception as e:
            print(f"Impossible de lire le fichier {file_path} : {e}")
                    
    print(f"\n Chargement terminé ! Total de {len(documents)} Documents récupérés.")
    return documents

## Load document and chunks

In [19]:
# writes pages that have not been loaded yet as txt, and returns the new pages names
new_pages = scrape_web()

# reads the new pages content as Documents
all_docs = load_txt_documents(new_pages)

Chargement de wikipedia OK
FANDOM OK: Survie (10 sections)
Chargement de la page Survie OK
FANDOM OK: Créatif (6 sections)
Chargement de la page Créatif OK
FANDOM OK: Hardcore (3 sections)
Chargement de la page Hardcore OK
FANDOM OK: Fabrication (4 sections)
Chargement de la page Fabrication OK
FANDOM OK: Cuisson (7 sections)
Chargement de la page Cuisson OK
FANDOM OK: Alchimie (9 sections)
Chargement de la page Alchimie OK
FANDOM OK: La_Foire_aux_Questions (14 sections)
Chargement de la page La_Foire_aux_Questions OK
FANDOM OK: Tutoriels/Choses_à_ne_PAS_faire (24 sections)
Chargement de la page Tutoriels/Choses_à_ne_PAS_faire OK
FANDOM OK: Tutoriels/Survie_dans_le_Nether (17 sections)
Chargement de la page Tutoriels/Survie_dans_le_Nether OK
FANDOM OK: Tutoriels/L'End_et_l'Ender_Dragon (8 sections)
Chargement de la page Tutoriels/L'End_et_l'Ender_Dragon OK
FANDOM OK: Créatures (17 sections)
Chargement de la page Créatures OK
Lecture du fichier : files/wikipedia.txt
Lecture du fichier :

### Chunking

In [20]:
def split_docs(all_docs):
    # Sécurité : Si la liste est vide, on s'arrête gentiment ici sans crasher
    if not all_docs:
        print("ANCIENS DOCS (Paragraphes bruts) : 0")
        print("NOUVEAUX CHUNKS ÉQUILIBRÉS : 0")
        print("Aucun nouveau document à découper (Tout est déjà à jour).")
        return []

    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1800, 
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    split_docs = splitter.split_documents(all_docs)

    print("ANCIENS DOCS (Paragraphes bruts) :", len(all_docs))
    print("NOUVEAUX CHUNKS ÉQUILIBRÉS :", len(split_docs))

    tailles = [len(doc.page_content) for doc in split_docs]
    if tailles:
        print(f"Taille Min : {min(tailles)} | Taille Max : {max(tailles)} | Moyenne : {int(sum(tailles)/len(tailles))}")
    
    return split_docs

In [21]:
all_docs = split_docs(all_docs)



ANCIENS DOCS (Paragraphes bruts) : 136
NOUVEAUX CHUNKS ÉQUILIBRÉS : 173
Taille Min : 130 | Taille Max : 1791 | Moyenne : 873


## Summarize + Initialize embedding model

In [ ]:
## ==========================================
## MULTI-VECTOR STORAGE (CHROMA + DISK STORE)
## ==========================================

# 1. Définition des chemins sur le disque
CHROMA_DIR = "./chroma_minecraft_multivec"
STORE_DIR = "./local_chunks_store"

# 2. Initialisation de la base vectorielle Chroma (Contient les résumés)
vectorstore = Chroma(
    collection_name="minecraft_summaries",
    persist_directory=CHROMA_DIR,
    embedding_function=gemini_embeddings
)

# 3. Initialisation du stockage persistant sur DISQUE (Contient les chunks bruts)
fs_store = LocalFileStore(STORE_DIR)

# 4. Création du Retriever Multi-Vecteurs
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=fs_store,
    id_key="doc_id",
)
# ==========================================
# LOGIQUE D'INDEXATION ÉCONOMIQUE (ANTI-DUPLICATA)
# ==========================================

# Vérifier ce qui est déjà présent dans Chroma
try:
    existing_data = vectorstore.get()
    # Clé composite unique (source + section) pour éviter les collisions entre pages différentes
    existing_sections = set(
        f"{meta.get('source')}_{meta.get('section')}" 
        for meta in existing_data.get("metadatas", []) if meta
    )
    print(f"Base de données trouvée : {len(existing_sections)} sections uniques déjà présentes dans Chroma.")
except Exception:
    existing_sections = set()
    print("Base vectorielle vierge ou inaccessible. Première indexation.")

# Filtrer les documents récupérés avec la même clé composite unique
docs_to_index = [
    doc for doc in all_docs 
    if f"{doc.metadata.get('source')}_{doc.metadata.get('section')}" not in existing_sections
]

if not docs_to_index:
    print("Tous les documents sont déjà à jour sur le disque. ZÉRO crédit consommé ! ")
else:
    print(f"{len(docs_to_index)} nouveaux chunks à traiter (Génération Ollama + Embedding Gemini)...")
    
    # La boucle traite ET sauvegarde en temps réel désormais
    for i, doc in enumerate(docs_to_index):
        text_len = len(doc.page_content)
        print(f"[{i+1}/{len(docs_to_index)}] Analyse du chunk ({text_len} caractères)...", end="")
        
        # Génération du résumé via le modèle local Ollama
        if text_len < 700:
            summary_text = doc.page_content
        else:
            try:
                summary_text = summarize_chain.invoke({"doc": doc.page_content}).strip() 
            except Exception as e:
                print(f"\nErreur Ollama sur le chunk {i}: {e}")
                summary_text = doc.page_content
        
        # Liaison unique entre le résumé et le gros chunk
        chunk_id = str(uuid.uuid4())
        
        summary_doc = Document(
            page_content=summary_text,
            metadata={**doc.metadata, "doc_id": chunk_id}
        )
        
        # --- SAUVEGARDE EN TEMPS RÉEL (Sécurisée par un try/except) ---
        try:
            # 1. On envoie le résumé tout de suite dans Chroma
            retriever.vectorstore.add_documents([summary_doc])
            
            # 2. On sauvegarde le chunk brut tout de suite sur le disque local
            retriever.docstore.mset([(chunk_id, doc)])
            
            print(" -> Sauvegardé ! ")
        except Exception as e:
            print(f"\nErreur de connexion/sauvegarde sur le chunk {i} (Gemini déconnecté ?) : {e}")
            print("Arrêt de sécurité. Relance la cellule quand ton réseau sera stable.")
            break

# --- Vérification finale des volumes ---
print("\n--- STATISTIQUES ---")
print("Total Chunks générés ce run (split_docs):", len(all_docs))
print("Total Résumés actuellement dans Chroma:", vectorstore._collection.count())

C:\Users\chahi\AppData\Local\Temp\ipykernel_18664\2326982033.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Base de données trouvée : 0 sections uniques déjà présentes dans Chroma.
173 nouveaux chunks à traiter (Génération Ollama + Embedding Gemini)...
[1/173] Analyse du chunk (1662 caractères)...
[2/173] Analyse du chunk (482 caractères)...
[3/173] Analyse du chunk (1382 caractères)...
[4/173] Analyse du chunk (1167 caractères)...
[5/173] Analyse du chunk (1174 caractères)...
[6/173] Analyse du chunk (1614 caractères)...
[7/173] Analyse du chunk (1646 caractères)...
[8/173] Analyse du chunk (1699 caractères)...
[9/173] Analyse du chunk (1546 caractères)...
[10/173] Analyse du chunk (688 caractères)...
[11/173] Analyse du chunk (402 caractères)...
[12/173] Analyse du chunk (1207 caractères)...
[13/173] Analyse du chunk (1205 caractères)...
[14/173] Analyse du chunk (1116 caractères)...
[15/173] Analyse du chunk (803 caractères)...
[16/173] Analyse du chunk (955 caractères)...
[17/173] Analyse du chunk (839 caractères)...
[18/173] Analyse du chunk (969 caractères)...
[19/173] Analyse du chunk

## Retriever

#### Create a retriever using Chroma

In [ ]:
print(retriever.invoke("Minecraft")[0].page_content[:500])

def retrieve_documents(retriever, query):
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    if hasattr(retriever, "get_relevant_documents"):
        return retriever.get_relevant_documents(query)
    raise AttributeError("Le retriever ne supporte ni invoke ni get_relevant_documents.")

Minecraft plonge le joueur dans un monde créé de manière procédurale, composé de voxels représentant différents matériaux ( terre , pierre , eau , fer , charbon , etc. ). Le monde est formé de diverses structures ( arbres , cavernes , montagnes , villages , etc. ) et est peuplé par des animaux ( vaches , moutons , etc. ) ainsi que des monstres ( zombies , araignées , squelettes , etc. ). Le joueur peut modifier son monde à volonté, soit dans le but de survivre, soit pour créer.


## Generator

#### Create prompt templates

In [ ]:
# Prompt template （more strict） to query Gemini
llm_prompt_template = """Tu es un assistant expert sur le jeu Minecraft.
Réponds à la question en utilisant UNIQUEMENT le contexte fourni ci-dessous.
Si la réponse ne se trouve pas dans le contexte ou si tu n'es pas sûr, n'invente rien. Dis EXACTEMENT : "Je suis désolé, mais l'information n'est pas dans les documents fournis."
Sois concis (5 phrases maximum) et cite la ou les sources [SOURCE_TYPE: source] à la fin de ta réponse si tu as trouvé l'information.

Question: {question}
Contexte: {context}
Réponse:"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template='Tu es un assistant expert sur le jeu Minecraft.\nRéponds à la question en utilisant UNIQUEMENT le contexte fourni ci-dessous.\nSi la réponse ne se trouve pas dans le contexte ou si tu n\'es pas sûr, n\'invente rien. Dis EXACTEMENT : "Je suis désolé, mais l\'information n\'est pas dans les documents fournis."\nSois concis (5 phrases maximum) et cite la ou les sources [SOURCE_TYPE: source] à la fin de ta réponse si tu as trouvé l\'information.\n\nQuestion: {question}\nContexte: {context}\nRéponse:'


#### Create a stuff documents chain

In [ ]:
# Combine data from documents to readable string format.
def format_docs(docs):
    formatted_docs = []

    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        content = doc.page_content

        formatted_text = f"[{source}]\n{content}"

        formatted_docs.append(formatted_text)

    return "\n\n".join(formatted_docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    | llm
    | StrOutputParser()
)

### RAG AVANCÉ : ACTIVE RETRIEVAL (RRR)

In [ ]:
def check_if_answered(reponse):
    prompt = f"La réponse suivante indique-t-elle qu'elle ne trouve pas l'information ou qu'elle ne sait pas ? Réponds OUI ou NON.\nRéponse : {reponse}"
    result = llm.invoke(prompt)
    text_content = str(result.content)
    return "OUI" in text_content.strip().upper()

def ask_with_active_retrieval(question):
    print(f"▶ Question posée : {question}")
    
    reponse = rag_chain.invoke(question)
    phrase_echec = "Je suis désolé, mais l'information n'est pas dans les documents fournis." # Phrase exite dans llm_prompt_template，à détecter pour déclencher l'auto-correction
    
    # Auto-correction
    if check_if_answered(reponse):
        print("Information introuvable. Activation de l'Active Retrieval...")
        
        # 1. IA de réécriture (Query Optimizer)
        llm_rewrite = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
        rewrite_prompt = f"""Tu es un expert Minecraft. Un utilisateur a posé cette question : '{question}'. 
        Cette question n'a donné aucun résultat exact dans notre base de données RAG. 
        Réécris cette question sous forme de 2 ou 3 mots-clés très simples et percutants pour optimiser une recherche par similarité vectorielle. 
        Ne renvoie QUE les mots-clés (par exemple : 'Ender Dragon stratégie'), rien d'autre."""
        
        # Générer la nouvelle requête
        nouvelle_requete = str(llm_rewrite.invoke(rewrite_prompt).content)
        print(f"Nouvelle requête optimisée par l'IA : {nouvelle_requete}")
        
        # 2. Relancer la recherche avec les nouveaux mots-clés
        reponse_niveau_2 = rag_chain.invoke(nouvelle_requete)
        
        # 3. Vérifier si la deuxième tentative a réussi
        if phrase_echec not in reponse_niveau_2:
            print("Succès du Niveau 2.")
            return f"*(Auto-correction RRR : Recherche élargie avec les mots-clés '{nouvelle_requete}')*\n\n{reponse_niveau_2}"
        else:
            print("Échec du Niveau 2. L'information n'existe définitivement pas dans la base.")
            return reponse # Renvoie le message de refus standard
            
    # Si le niveau 1 a marché directement
    print("Succès du Niveau 1.")
    return reponse

### Prompt the model

In [ ]:
Markdown(ask_with_active_retrieval("Quel est l'ingrédient de base indispensable pour l'alchimie ?"))

▶ Question posée : Quel est l'ingrédient de base indispensable pour l'alchimie ?


ConnectError: [WinError 10061] Aucune connexion n’a pu être établie car l’ordinateur cible l’a expressément refusée

In [ ]:
Markdown(ask_with_active_retrieval("C'est quoi le Warden dans Minecraft ?"))

In [ ]:
Markdown(ask_with_active_retrieval("Comment survivre dans le Nether dans Minecraft ?"))

In [ ]:
Markdown(ask_with_active_retrieval("C'est quoi l'alchimie dans Minecraft?"))

In [ ]:
Markdown(ask_with_active_retrieval("Il y a combien de dimensions dans Minecraft? Décris les."))

In [ ]:
Markdown(ask_with_active_retrieval("Parle moi des boss dans Minecraft, et décris les tous."))

In [ ]:
### ÉVALUATION DU RAG (LLM-as-a-Judge)